## Importando bibliotecas e dataset

In [1]:
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    classification_report,
    confusion_matrix
)

paths = glob.glob("/kaggle/input/datasets/menegidio/iris-species/Iris.csv", recursive=True)

if not paths:
    raise FileNotFoundError(
        "Iris.csv não foi encontrado. No Kaggle, use Add Data e adicione o dataset Iris Species."
    )

DATA_PATH = paths[0]
df = pd.read_csv(DATA_PATH)

print("Dimensões:", df.shape)
display(df.head())

Dimensões: (150, 6)


,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


## Informações gerais

In [ ]:
features = [
    "SepalLengthCm",
    "SepalWidthCm",
    "PetalLengthCm",
    "PetalWidthCm"
]
target = "Species"

X = df[features]
y = df[target]

print("Classes:")
display(y.value_counts())

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste :", X_test.shape)

## Boosting — Gradient Boosting



In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

modelo = GradientBoostingClassifier(random_state=42)

param_grid = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [1, 2, 3]
}

In [ ]:
def avaliar_modelo(modelo, nome, X_test, y_test):
    pred = modelo.predict(X_test)

    acc = accuracy_score(y_test, pred)
    sens = recall_score(y_test, pred, average="macro")

    print(f"Modelo: {nome}")
    print(f"Acurácia: {acc:.4f}")
    print(f"Sensibilidade (macro recall): {sens:.4f}")
    print("\nRelatório de classificação:")
    print(classification_report(y_test, pred, digits=4))

    cm = confusion_matrix(y_test, pred, labels=sorted(y.unique()))
    plt.figure(figsize=(6, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=sorted(y.unique()),
        yticklabels=sorted(y.unique())
    )
    plt.title(f"Matriz de confusão - {nome}")
    plt.xlabel("Predito")
    plt.ylabel("Real")
    plt.show()

    return {
        "Modelo": nome,
        "Configuração": "Sem GridSearchCV",
        "Acurácia": acc,
        "Sensibilidade": sens
    }

modelo.fit(X_train, y_train)
resultado_padrao = avaliar_modelo(
    modelo,
    "Boosting - padrão",
    X_test,
    y_test
)

grid = GridSearchCV(
    estimator=modelo,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    return_train_score=True
)

grid.fit(X_train, y_train)

print("Melhores hiperparâmetros encontrados:")
print(grid.best_params_)
print(f"Melhor acurácia média na validação cruzada: {grid.best_score_:.4f}")

melhor_modelo = grid.best_estimator_
pred_grid = melhor_modelo.predict(X_test)

acc_grid = accuracy_score(y_test, pred_grid)
sens_grid = recall_score(y_test, pred_grid, average="macro")

print(f"\nAcurácia no teste com GridSearchCV: {acc_grid:.4f}")
print(f"Sensibilidade no teste com GridSearchCV: {sens_grid:.4f}")
print("\nRelatório de classificação:")
print(classification_report(y_test, pred_grid, digits=4))

cm_grid = confusion_matrix(y_test, pred_grid, labels=sorted(y.unique()))
plt.figure(figsize=(6, 4))
sns.heatmap(
    cm_grid,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=sorted(y.unique()),
    yticklabels=sorted(y.unique())
)
plt.title(f"Matriz de confusão - Boosting com GridSearchCV")
plt.xlabel("Predito")
plt.ylabel("Real")
plt.show()

resultado_grid = {
    "Modelo": "Boosting",
    "Configuração": "Com GridSearchCV",
    "Acurácia": acc_grid,
    "Sensibilidade": sens_grid
}

comparacao = pd.DataFrame([resultado_padrao, resultado_grid])
display(comparacao)

## Conclusão

Na comparação dos modelos, os melhores resultados de acurácia e sensibilidade foram de 96,67%. Esse desempenho foi alcançado pelo SVM sem GridSearchCV, pela Floresta Aleatória com GridSearchCV e pelo Boosting com e sem GridSearchCV. A Árvore de Decisão apresentou 93,33% nas duas configurações. Portanto, os resultados mostram que SVM, Floresta Aleatória e Boosting tiveram o mesmo maior desempenho observado no experimento, enquanto a Árvore de Decisão apresentou um resultado um pouco inferior. Não é possível dizer qual modelo é o mais preciso ou tem a melhor sensibilidade, já que vários deles apresentam resultados semelhantes ou iguais.